# Step 2 — Ring-geometry inference with **dynesty** (nested sampling)

Builds a joint KDE likelihood from the step-1 transit observables and samples the ring geometry
$(f_e,\ i_R,\ \theta,\ p,\ \dots)$ with static nested sampling, which also yields the Bayesian
evidence $\ln\mathcal{Z}$. The model, likelihood and priors live in
[`photoring`](photoring/) ([`PhotoRingModel`](photoring/model.py)); this notebook wires a *case*
to a *run configuration* and drives the sampler.

**Configure a run** in the `USER CONFIGURATION` cell (§1). Everything downstream is automatic.
The same cell is `papermill`-parametrised, so `run_sweep.py` can sweep many configurations.

| Observable key | meaning |
|---|---|
| `delta`, `T14`, `T23` | depth, total / flat contact durations |
| `rho_obs` | transit-inferred stellar density |
| `b_obs` | transit-inferred impact parameter |

## 0. Environment

In [ ]:
# ── Bootstrap: make the sibling packages importable without installation ────
# The pipeline uses three packages that live in the repository, uninstalled:
#   exorings, geotrans   (repo root)      photoring   (pipeline/)
# We locate the repo root and pipeline/ robustly from the current working dir
# (Jupyter / nbconvert / papermill all run notebooks from pipeline/).
import sys, pathlib
_HERE   = pathlib.Path.cwd().resolve()
_cands  = [_HERE, *_HERE.parents]
_NB_DIR = next((c for c in _cands if (c / "photoring").is_dir()), _HERE)
_REPO   = next((c for c in _cands if (c / "exorings").is_dir()), _NB_DIR.parent)
for _p in (str(_REPO), str(_NB_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root :", _REPO)
print("pipeline  :", _NB_DIR)

In [ ]:
import numpy as np
import warnings, os, time
warnings.filterwarnings("ignore")
# Limit BLAS threads before heavy numerics (pool workers each add threads).
for _v in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_v, "2")

import photoring as pr
import photoring.plotting as plot
plot.apply_style()

import dynesty
from dynesty import plotting as dyplot
print("dynesty", dynesty.__version__)

## 1. USER CONFIGURATION  ← edit here (papermill-injected)

In [ ]:
# All entries must be plain literals (papermill injects scalars/lists/dicts here).
CASE   = "kepler_51"
PLANET = "d"            # papermill overrides -> 'b' or 'd'

# Planet-specific priors (Masuda 2024). Resolved into MODEL_CONFIG below.
PLANET_PARAMS = {
    "d": dict(B_FIXED=0.0030, B_SIGMA=2 * 0.0950, p_mean_ref=0.09857, p_prior_lo=0.23),
    "b": dict(B_FIXED=0.0740, B_SIGMA=2 * 0.0720, p_mean_ref=0.07225, p_prior_lo=0.33),
}

KDE_CONFIG = {"observables": ["delta", "rho_obs", "T14"], "N_KDE": 5000, "seed_kde": 123}

NS_CONFIG = {"nlive": 1200, "sample": "rslice", "dlogz": 0.01, "bound": "multi",
             "seed": 2026, "use_pool": True, "n_procs": 6}

MODEL_CONFIG = {
    "B_FREE": False, "B_FIXED": PLANET_PARAMS[PLANET]["B_FIXED"],
    "B_SIGMA": PLANET_PARAMS[PLANET]["B_SIGMA"],
    "RHO_TRUE_FREE": True, "RHO_TRUE_FIXED": None,
    "FI_FIXED": 1.0, "FE_MAX": 10.0,
    "TAU_FREE": False, "TAU_FIXED": 1.0, "TAU_PRIOR_LO": 0.1, "TAU_PRIOR_HI": 10.0,
    "P_FREE": True, "p_mean_ref": PLANET_PARAMS[PLANET]["p_mean_ref"],
    "p_prior_lo": PLANET_PARAMS[PLANET]["p_prior_lo"], "p_prior_hi": 1.0,
    "FORWARD_MODEL": "exorings",   # "exorings" (closed-form) or "geotrans" (numeric)
}

## 2. Build the model (data + KDE likelihood + priors)

In [ ]:
paths = pr.CasePaths(CASE)
FORWARD_MODEL = str(MODEL_CONFIG.get("FORWARD_MODEL", "exorings")).lower()
paths.ensure_outputs(FORWARD_MODEL)

# Load the case's derived observables, rho_true samples and inverse-CDF grid.
data  = pr.load_case_data(paths, PLANET)
model = pr.PhotoRingModel(
    data["ttv"], data["rho_true_gcc_samples"], MODEL_CONFIG, KDE_CONFIG,
    rho_grid=data["rho_grid"], rho_cdf=data["rho_cdf"], p_fixed=data["P_fixed"],
)
print(f"Planet {PLANET}: {len(data['ttv']['delta'])} TTV samples | P_fixed={data['P_fixed']:.6f} d")
print(f"NDIM={model.NDIM}  params={model.PARAM_NAMES}")

In [ ]:
# Reproducible run tag encoding the full configuration.
_kt = "-".join(model.observables)
RUN_TAG = (f"{CASE}_{PLANET}_NS_{FORWARD_MODEL}_kde_{_kt}"
           f"_nlive{NS_CONFIG['nlive']}_dlogz{NS_CONFIG['dlogz']}"
           f"_NKDE{KDE_CONFIG['N_KDE']}_seed{NS_CONFIG['seed']}{model.free_tag()}")
print("RUN_TAG:", RUN_TAG)

## 3. KDE self-consistency check

In [ ]:
plot.plot_kde_ppc(model, planet=PLANET, paths=paths, run_tag=RUN_TAG); import matplotlib.pyplot as plt; plt.show()

## 4. Nested sampling

In [ ]:
import multiprocessing as mp
try:    _ctx = mp.get_context("fork")
except ValueError: _ctx = mp

result = pr.run_dynesty(model, NS_CONFIG, ctx=_ctx)
print(f"\nlnZ = {result['logz']:.3f} +/- {result['logz_err']:.3f}"
      f"  | runtime {result['runtime_s']:.1f}s | N={len(result['chain'])}")

## 5. Posterior summary

In [ ]:
for name, lbl in zip(model.PARAM_NAMES, model.PARAM_LABELS):
    s = result["stats"][name]; m = s["median"]
    print(f"  {name:>10}: {m:.5f}  [-{m-s['p16']:.5f}, +{s['p84']-m:.5f}]")

## 6. Posterior predictive check (all observables)

In [ ]:
import matplotlib.pyplot as plt
ppc = pr.compute_ppc(model, result["chain"])
run = pr.make_run(model, result, RUN_TAG, PLANET, ppc=ppc)
plot.plot_ppc(run, data["ttv"], paths=paths); plt.show()

## 7. dynesty native diagnostics (run / trace plots)

In [ ]:
import matplotlib.pyplot as plt
dres = result["dres"]
fig_run, _ = dyplot.runplot(dres, lnz_error=True)
fig_run.savefig(paths.figures_dir("trace") / f"{RUN_TAG}_runplot.png", dpi=plot.STYLE["fig_dpi"]); plt.show()
fig_tr, _ = dyplot.traceplot(dres, labels=model.PARAM_LABELS, show_titles=True, trace_cmap="plasma")
fig_tr.savefig(paths.figures_dir("trace") / f"{RUN_TAG}_traceplot.png", dpi=plot.STYLE["fig_dpi"]); plt.show()

## 8. Marginals and corner (publication style)

In [ ]:
import matplotlib.pyplot as plt
plot.plot_marginals(run, berger_rho=data["rho_true_gcc_samples"], paths=paths); plt.show()
plot.plot_corner(run, paths=paths); plt.show()

## 9. Save results

In [ ]:
meta = dict(
    planet=PLANET, case=CASE, run_tag=RUN_TAG, sampler="dynesty",
    kde_observables=model.observables, N_KDE=int(len(model.idx_train)),
    seed_kde=int(KDE_CONFIG["seed_kde"]),
    nlive=int(NS_CONFIG["nlive"]), sample=NS_CONFIG["sample"], dlogz=float(NS_CONFIG["dlogz"]),
    seed_ns=int(NS_CONFIG["seed"]), FORWARD_MODEL=FORWARD_MODEL,
    B_FREE=bool(model.B_FREE), B_FIXED=float(model.B_FIXED), B_SIGMA=float(model.B_SIGMA),
    RHO_TRUE_FREE=bool(model.RHO_TRUE_FREE), RHO_TRUE_FIXED=float(model.RHO_TRUE_FIXED),
    TAU_FREE=bool(model.TAU_FREE), TAU_FIXED=float(model.TAU_FIXED),
    P_FREE=bool(model.P_FREE), FI_FIXED=float(model.FI_FIXED), FE_MAX=float(model.FE_MAX),
    p_min=float(model.p_min), p_max=float(model.p_max), p_mean_ref=float(model.p_mean_ref),
    P_fixed_days=float(model.P_fixed),
    logz=float(result["logz"]), logz_err=float(result["logz_err"]),
    n_iter=int(result["n_iter"]), runtime_s=float(result["runtime_s"]),
    n_samples=int(len(result["chain"])), param_names=model.PARAM_NAMES,
)
for _n, _s in result["stats"].items():
    meta[f"stat_{_n}_median"] = float(_s["median"]); meta[f"stat_{_n}_p16"] = float(_s["p16"]); meta[f"stat_{_n}_p84"] = float(_s["p84"])

arrays = dict(chain=result["chain"], samples=dres.samples, logwt=dres.logwt, logl=dres.logl, ppc=ppc)
pr.save_run(paths.results_dir(FORWARD_MODEL), RUN_TAG, arrays, meta)
print("Saved ->", paths.results_dir(FORWARD_MODEL))